## Extract the complexity measures 

In [1]:
import os, random, sys, warnings

In [2]:
import numpy as np
import pandas as pd

In [3]:
from multiprocessing import Pool
from pymfe.mfe import MFE

In [4]:
sys.path.append('../src')
from kumo_prior import sample_datasets

In [5]:
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [6]:
SEED = 42

In [7]:
FEATURES = ['f1', 'f1v', 'f2', 'f3', 'f4', 'l1', 'l2', 'l3', 'n1', 'n2', 'n3', 't2', 't3', 't4', 'c1', 'c2']

## Reading parameters

In [8]:
mode = os.environ['MODE']
jobs = int(os.environ['JOBS'])

if mode == 'synthetic':
    stage = int(os.environ['STAGE'])
    n_datasets = int(os.environ['DATASETS'])

In [9]:
random.seed(SEED)
np.random.seed(SEED)

## Helper functions

In [10]:
def normalize(file):
    df = pd.read_csv(file, na_values=['?', 'Not_Say'], low_memory=False)
    X, y = df.iloc[:, :-1], df.iloc[:, -1]

    X = X.dropna(axis=1, how='all')
    cat = X.select_dtypes(exclude='number').columns
    num = X.columns.difference(cat)

    X[num] = X[num].fillna(X[num].mean())
    X[cat] = X[cat].fillna('missing')

    return X.to_numpy(), y.to_numpy()

In [11]:
def load_datasets(path):
    return [(file, *normalize(f'{path}/{file}')) for file in os.listdir(path)]

In [12]:
def extract(args):
    i, X, y = args
    mfe = MFE(groups='complexity', features=FEATURES, summary='nanmean', random_state=SEED)
    mfe.fit(X, y, precomp_groups=None, suppress_warnings=True)
    measures, values = mfe.extract(suppress_warnings=True)
    return {'id': i, 'n_classes': len(np.unique(y)), 'n_rows': X.shape[0], 'n_cols': X.shape[1], 
            **dict(zip(measures, map(float, values)))}

In [13]:
def run(data, out, jobs):
    done = set(pd.read_csv(out, usecols=['id'])['id'].astype(str)) if os.path.exists(out) else set()
    columns = list(pd.read_csv(out, nrows=0).columns) if done else None
    todo = (t for t in data if str(t[0]) not in done)

    with Pool(jobs) as pool:
        for row in pool.imap_unordered(extract, todo):
            columns = columns or list(row)
            pd.DataFrame([row], columns=columns).to_csv(out, mode='a', header=not os.path.exists(out), index=False)

## Generating real datasets

In [15]:
if mode == 'real':
    data = load_datasets('../data/processed/tabarena/') + load_datasets('../data/processed/talent/')
    run(data, f'../results/{mode}.csv', jobs)

## Generating synthetic datasets

In [18]:
NO_MISSING = dict(structured_missingness_prob=0, mcar_missingness_prob=0, near_empty_missingness_prob=0, latent_driver_missingness_prob=0)

In [22]:
if mode == 'synthetic':
    synthetic = sample_datasets('clf', stage=stage, n_datasets=n_datasets, seed=SEED, size='small', 
                                batch_size=1, group_size=1, **NO_MISSING)
    data = ((i, d['X'], d['y']) for i, d in enumerate(synthetic))
    run(data, f'../results/kumo_stage{stage}.csv', jobs)